---
title: "Pandas: DataFrames, Series and the way over from SQL"
short_title: "Pandas"
description: "DataFrame and Series as data structures, the comparison with SQL and relational databases, the translation of typical queries, and a first look at a dataset."
---

<!---
https://pbpython.com/categorical-encoding.html
--->

## DataFrame and Series

The **DataFrame** is the data structure everything in Pandas revolves around: a table with named columns, each column with its own data type. If you know R, you know it as `data.frame`; if you have only used Numpy so far, think of it as a record array (`np.genfromtxt(..., names=True)` in the chapter [Exploratory analysis](eda-examples.ipynb) produces one of those), only more convenient. The nearest thing to it is an Excel sheet in Python.

Unlike a Numpy array, a DataFrame is always two-dimensional – there is no such thing as a DataFrame with three or four axes. That is not a restriction but a commitment to exactly the shape data usually arrive in: rows are observations, columns are features.

In [1]:
import numpy as np
import pandas as pd

columns = {
    "species": ["setosa", "setosa", "versicolor"],
    "sepal_width": [3.5, 3.0, 3.2],
}
df = pd.DataFrame(columns)
print(df)

      species  sepal_width
0      setosa          3.5
1      setosa          3.0
2  versicolor          3.2


There are two ways to get at a column, and they do the same thing: with square brackets, as for a dictionary, or as an attribute. The brackets always work; the attribute notation only works if the column name is a valid Python identifier and does not happen to shadow a method – so when in doubt, brackets.

In [2]:
print(df["sepal_width"])
print(df.sepal_width)  # the same thing

print(df.dtypes)  # here you can see that Numpy is underneath
print(df["sepal_width"].to_numpy())

0    3.5
1    3.0
2    3.2
Name: sepal_width, dtype: float64
0    3.5
1    3.0
2    3.2
Name: sepal_width, dtype: float64
species            str
sepal_width    float64
dtype: object
[3.5 3.  3.2]


A single column is not a DataFrame but the second of the two data structures Pandas has: a **Series**, that is, a one-dimensional array with an index. Most methods exist for both, some only for one of the two – the error message `'Series' object has no attribute ...` usually means you have accidentally got hold of a column instead of the table.

To disk and back is `to_csv` and `read_csv`; the latter also accepts a URL. The `index=False` when writing matters: without it the row index lands in the file as a column and comes back as `Unnamed: 0` when the file is read in again.

In [3]:
print(type(df["sepal_width"]))

df.to_csv("example.csv", index=False, encoding="utf-8")
back_again = pd.read_csv("example.csv")
print(back_again)

import os
os.remove("example.csv")

<class 'pandas.Series'>
      species  sepal_width
0      setosa          3.5
1      setosa          3.0
2  versicolor          3.2


Pandas has some of the best documentation in the Python ecosystem, and it is worth using instead of guessing: the [tutorials for getting started](https://pandas.pydata.org/docs/getting_started/tutorials.html), the API reference with a page per method (`pandas.DataFrame.groupby`, say) and the [overview of the data types](https://pandas.pydata.org/docs/reference/arrays.html).

That last one is worth a look because the Pandas type system goes beyond the Numpy types. Besides `int64`, `float64` and `bool` there are, among others:

| Kind of data | pandas data type | Scalar |
|---|---|---|
| point in time with time zone | `DatetimeTZDtype` | `Timestamp` |
| duration | – | `Timedelta` |
| period | `PeriodDtype` | `Period` |
| interval | `IntervalDtype` | `Interval` |
| integer with missing values | `Int64Dtype`, … | – |
| categorical values | `CategoricalDtype` | – |
| sparse column | `SparseDtype` | – |
| strings | `StringDtype` | `str` |
| truth values with missing values | `BooleanDtype` | – |

Two of them will turn up again: `CategoricalDtype` in one-hot encoding in [Regression analysis](linear-regression.ipynb) and the timestamps in [Data cleaning](data-cleaning.ipynb). That the column `species` above has type `str` rather than `object` is new – up to pandas 2, strings lived in an `object` column, which you will still see often in older code and older tutorials.

## SQL and relational databases

Anything that can be expressed in SQL can be expressed with DataFrames, and the rest of this chapter puts the two side by side. A few words on where that language fits are in order first.

**SQL** is the standard query language for relational database systems (RDBMS) and has been an ISO standard since 1987. The two most widespread free systems are MySQL and PostgreSQL; they sit behind a large share of the smaller websites – every WordPress installation comes with a database, and WordPress powers something like 40% of all websites. Anyone pulling data out of a live system almost always writes SQL first and only then carries on in Python.

The most important commands, grouped by what they touch:

| Command | Effect |
|---|---|
| `SELECT` | select data and output it |
| `INSERT INTO` | add rows |
| `UPDATE` | change existing values |
| `DELETE` | delete rows |
| `CREATE` / `ALTER DATABASE` | create, change a database |
| `CREATE` / `ALTER` / `DROP TABLE` | create, change, delete a table |
| `CREATE` / `DROP INDEX` | create, delete a search key |

A query then looks like this, for example:

```sql
SELECT "Petal.Width" FROM iris;
```

The vocabulary of the relational world is worth knowing, because you meet it constantly in documentation and error messages:

:::{admonition} Definition
A **database schema** is a finite, non-empty set of **relation schemas**; a relation schema is the formal description of the data types of a **relation**. A relation is what you see as a **table**: its rows are called **tuples** or *entities*, its columns are called **attributes**, and the values permitted for an attribute form its **domain**.
:::

Translated into the Pandas world:

| relational | Pandas |
|---|---|
| relation / table | `DataFrame` |
| attribute / column | `Series` |
| tuple / row | a row, addressed via the index |
| domain | the `dtype` of the column |

One difference remains: mathematically, a relation is a *set* of tuples and so has no order, whereas a DataFrame keeps its rows in order and addresses them through an index. That is why SQL needs an `ORDER BY` to get any order at all – and why Pandas does not guarantee the row order after a `merge`.

## From SQL to Pandas

The comparison below follows the [Pandas documentation, section "Comparison with SQL"](https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sql.html). In each code cell, the SQL statement appears as a comment above the Pandas expression that does the same job.

In [4]:
import numpy as np
import pandas as pd

url = (
    "https://raw.github.com/pandas-dev"
    "/pandas/main/pandas/tests/io/data/csv/tips.csv"
)

tips = pd.read_csv(url)
print(tips)

     total_bill   tip     sex smoker   day    time  size
0         16.99  1.01  Female     No   Sun  Dinner     2
1         10.34  1.66    Male     No   Sun  Dinner     3
2         21.01  3.50    Male     No   Sun  Dinner     3
3         23.68  3.31    Male     No   Sun  Dinner     2
4         24.59  3.61  Female     No   Sun  Dinner     4
..          ...   ...     ...    ...   ...     ...   ...
239       29.03  5.92    Male     No   Sat  Dinner     3
240       27.18  2.00  Female    Yes   Sat  Dinner     2
241       22.67  2.00    Male    Yes   Sat  Dinner     2
242       17.82  1.75    Male     No   Sat  Dinner     2
243       18.78  3.00  Female     No  Thur  Dinner     2

[244 rows x 7 columns]


Operations on Pandas DataFrames produce copies; the original DataFrame stays unchanged.

In [5]:
tips.sort_values("total_bill")
print(tips)
print("\nThat is not sorted at all! But the next one is:\n")
sorted_tips = tips.sort_values("total_bill")
print(sorted_tips)

     total_bill   tip     sex smoker   day    time  size
0         16.99  1.01  Female     No   Sun  Dinner     2
1         10.34  1.66    Male     No   Sun  Dinner     3
2         21.01  3.50    Male     No   Sun  Dinner     3
3         23.68  3.31    Male     No   Sun  Dinner     2
4         24.59  3.61  Female     No   Sun  Dinner     4
..          ...   ...     ...    ...   ...     ...   ...
239       29.03  5.92    Male     No   Sat  Dinner     3
240       27.18  2.00  Female    Yes   Sat  Dinner     2
241       22.67  2.00    Male    Yes   Sat  Dinner     2
242       17.82  1.75    Male     No   Sat  Dinner     2
243       18.78  3.00  Female     No  Thur  Dinner     2

[244 rows x 7 columns]

That is not sorted at all! But the next one is:

     total_bill    tip     sex smoker   day    time  size
67         3.07   1.00  Female    Yes   Sat  Dinner     1
92         5.75   1.00  Female    Yes   Fri  Dinner     2
111        7.25   1.00  Female     No   Sat  Dinner     1
172       

Selecting columns is `SELECT`:

In [6]:
# SELECT total_bill, tip, smoker, time FROM tips;
tips[["total_bill", "tip", "smoker", "time"]]

,total_bill,tip,smoker,time
0,16.99,1.01,No,Dinner
1,10.34,1.66,No,Dinner
2,21.01,3.50,No,Dinner
3,23.68,3.31,No,Dinner
4,24.59,3.61,No,Dinner
...,...,...,...,...
239,29.03,5.92,No,Dinner
240,27.18,2.00,Yes,Dinner
241,22.67,2.00,Yes,Dinner
242,17.82,1.75,No,Dinner


You can add new columns computed from old ones:

In [7]:
# SELECT *, tip/total_bill as tip_rate FROM tips;
tips.assign(tip_rate=tips["tip"] / tips["total_bill"])

,total_bill,tip,sex,smoker,day,time,size,tip_rate
0,16.99,1.01,Female,No,Sun,Dinner,2,0.059447
1,10.34,1.66,Male,No,Sun,Dinner,3,0.160542
2,21.01,3.50,Male,No,Sun,Dinner,3,0.166587
3,23.68,3.31,Male,No,Sun,Dinner,2,0.139780
4,24.59,3.61,Female,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Female,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Male,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,Male,No,Sat,Dinner,2,0.098204


Filtering data (selecting rows by conditions) with `WHERE`

In [8]:
# SELECT * FROM tips WHERE time = 'Dinner';
is_dinner = tips["time"] == "Dinner" # bitmask
tips[is_dinner]

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.92,Male,No,Sat,Dinner,3
240,27.18,2.00,Female,Yes,Sat,Dinner,2
241,22.67,2.00,Male,Yes,Sat,Dinner,2
242,17.82,1.75,Male,No,Sat,Dinner,2


Combining several conditions logically with `OR |` or `AND &`:

In [9]:
# SELECT * FROM tips WHERE time = 'Dinner' AND tip > 6.00;
tips[(tips["time"] == "Dinner") & (tips["tip"] > 6.00)]

,total_bill,tip,sex,smoker,day,time,size
23,39.42,7.58,Male,No,Sat,Dinner,4
59,48.27,6.73,Male,No,Sat,Dinner,4
170,50.81,10.00,Male,Yes,Sat,Dinner,3
183,23.17,6.50,Male,Yes,Sun,Dinner,4
212,48.33,9.00,Male,No,Sat,Dinner,4
214,28.17,6.50,Female,Yes,Sat,Dinner,3


In [10]:
# SELECT * FROM tips WHERE size >= 6 OR total_bill > 45;
tips[(tips["size"] >= 6) | (tips["total_bill"] > 45)]

,total_bill,tip,sex,smoker,day,time,size
59,48.27,6.73,Male,No,Sat,Dinner,4
125,29.80,4.20,Female,No,Thur,Lunch,6
141,34.30,6.70,Male,No,Thur,Lunch,6
143,27.05,5.00,Female,No,Thur,Lunch,6
156,48.17,5.00,Male,No,Sun,Dinner,6
170,50.81,10.00,Male,Yes,Sat,Dinner,3
182,45.35,3.50,Male,Yes,Sun,Dinner,3
212,48.33,9.00,Male,No,Sat,Dinner,4


Split the data into groups, compute aggregate statistics, put the groups back together – that is `GROUP BY` in SQL and `groupby` in Pandas:

In [11]:
# SELECT sex, count(*) FROM tips GROUP BY sex;
tips.groupby("sex").size()
# tips.groupby("sex").count() would be wrong, that counts non-NA entries!

sex
Female     87
Male      157
dtype: int64

In [12]:
# SELECT day, AVG(tip), COUNT(*) FROM tips GROUP BY day;
tips.groupby("day").agg({"tip": "mean", "day": "size"})

,tip,day
day,,
Fri,2.734737,19
Sat,2.993103,87
Sun,3.255132,76
Thur,2.771452,62


A trap that snaps shut again and again when you aggregate: **the mean of the group means is not the
overall mean**, unless all the groups are the same size. The overall mean is the *weighted*
mean of the group means, with the group sizes as weights. With the values $10, 20, 30$ in
group A and $5, 15$ in group B, the group means are $20$ and $10$ and their mean is $15$ – but
the overall mean of all five values is $16$. So if all you have is a table of group means and
you want the overall mean from it, you need the `size` column as well. The same phenomenon in
a sharper form – group means whose order reverses when the groups are pooled – is called
*[Simpson's paradox](descriptive-statistics.md)*.


In [13]:
small = pd.DataFrame({"Group": ["A", "A", "A", "B", "B"], "Value": [10, 20, 30, 5, 15]})
group_means = small.groupby("Group")["Value"].mean()
print("mean of the group means:", group_means.mean())
print("overall mean:           ", small["Value"].mean())
# correct: weight by the group sizes
sizes = small.groupby("Group")["Value"].size()
print("weighted:               ", (group_means * sizes).sum() / sizes.sum())


mean of the group means: 15.0
overall mean:            16.0
weighted:                16.0


In [14]:
# SELECT smoker, day, COUNT(*), AVG(tip) FROM tips GROUP BY smoker, day;
tips.groupby(["smoker", "day"]).agg({"tip": ["size", "mean"]})

tip          
            size      mean
smoker day                
No     Fri     4  2.812500
       Sat    45  3.102889
       Sun    57  3.167895
       Thur   45  2.673778
Yes    Fri    15  2.714000
       Sat    42  2.875476
       Sun    19  3.516842
       Thur   17  3.030000

Data rarely arrive in a single table. Suppose the Iris measurements sit in two files: one holds the species of each plant, the other its sepal width. Both have the same number of rows – but may we simply lay them side by side? Only if the rows are in the same order, and you cannot know that once anyone has sorted one of the files in between.

So every row gets a column that identifies it uniquely: a **key**. That can be a plain number, or some unique piece of information that already exists (a student number, an ISBN). When data come together from several sources and nobody can hand out numbers centrally, you use a **UUID** (*universally unique identifier*), a 128-bit random number for which a collision is so unlikely that it can be ignored. The column that uniquely indexes a table is called its **primary key**; when the same key turns up in another table to point at those rows, it is a foreign key there. That is the basic idea of relational databases. The chapter [Relational data](https://r4ds.had.co.nz/relational-data.html) of *R for Data Science* has a fuller account.

In [15]:
import uuid

species = pd.DataFrame({"No": [1, 51], "species": ["setosa", "versicolor"]})
width = pd.DataFrame({"No": [51, 1], "sepal_width": [3.2, 3.5]})
print(species, "\n")
print(width, "\n")

# merged on the key No, so the order does not matter:
print(pd.merge(species, width, on="No"), "\n")

print(uuid.uuid4())  # a fresh UUID

   No     species
0   1      setosa
1  51  versicolor 

   No  sepal_width
0  51          3.2
1   1          3.5 

   No     species  sepal_width
0   1      setosa          3.5
1  51  versicolor          3.2 

2c741528-2614-4361-a4e1-b190d6217e53


It gets interesting when the two tables do not have the same keys. Then you have to decide what happens to the rows that find no partner – and that is exactly what separates the four kinds of join. The two tables below are built so that every case occurs: `A` exists only on the left, `E` only on the right, and `D` appears twice on the right.

In [16]:
df1 = pd.DataFrame({"key": ["A", "B", "C", "D"], "value": np.random.randn(4)})
df2 = pd.DataFrame({"key": ["B", "D", "D", "E"], "value": np.random.randn(4)})
print(df1, "\n", df2)

  key     value
0   A  0.370613
1   B  1.987784
2   C -0.885252
3   D  0.673109 
   key     value
0   B -1.594238
1   D  1.483229
2   D -0.353315
3   E  0.321249


In [17]:
# SELECT * FROM df1 INNER JOIN df2 ON df1.key = df2.key;
pd.merge(df1, df2, on="key") # inner join is the default

,key,value_x,value_y
0,B,1.987784,-1.594238
1,D,0.673109,1.483229
2,D,0.673109,-0.353315


The **inner join** keeps only the keys that occur in both tables: `B` and `D`. That `D` yields two rows is the most important point in the whole business – the key is not unique in `df2`, and a join then multiplies the rows on the other side. If a dataset is suddenly longer after a join than before, this is almost always where the problem lies.

In [18]:
# SELECT * FROM df1 LEFT OUTER JOIN df2 ON df1.key = df2.key;
pd.merge(df1, df2, on="key", how="left")

,key,value_x,value_y
0,A,0.370613,NaN
1,B,1.987784,-1.594238
2,C,-0.885252,NaN
3,D,0.673109,1.483229
4,D,0.673109,-0.353315


The **left outer join** keeps every row of the left table, partner or not: `A` and `C` come along too, with `NaN` in the columns from `df2`. This is the commonest case in practice – you have a dataset and want to enrich it with extra information without losing any rows along the way.

In [19]:
# SELECT * FROM df1 RIGHT OUTER JOIN df2 ON df1.key = df2.key;
pd.merge(df1, df2, on="key", how="right")

,key,value_x,value_y
0,B,1.987784,-1.594238
1,D,0.673109,1.483229
2,D,0.673109,-0.353315
3,E,NaN,0.321249


The **right outer join** is the same with the roles swapped: every row of `df2`, `E` included, and `NaN` where the left-hand side is missing.

In [20]:
# SELECT * FROM df1 FULL OUTER JOIN df2 ON df1.key = df2.key;
pd.merge(df1, df2, on="key", how="outer")

,key,value_x,value_y
0,A,0.370613,NaN
1,B,1.987784,-1.594238
2,C,-0.885252,NaN
3,D,0.673109,1.483229
4,D,0.673109,-0.353315
5,E,NaN,0.321249


The **full outer join** keeps everything, `A` to `E`, and pads both sides with `NaN` wherever nothing matches. Afterwards it is nearly always worth a look at `isna().sum()` to see how many gaps you have let yourself in for.

Changing data is `UPDATE` in SQL; in Pandas we simply run Python code, as if we were changing a single variable:

In [21]:
# state before the change:
tips.sort_values("tip").head()

,total_bill,tip,sex,smoker,day,time,size
92,5.75,1.00,Female,Yes,Fri,Dinner,2
111,7.25,1.00,Female,No,Sat,Dinner,1
67,3.07,1.00,Female,Yes,Sat,Dinner,1
236,12.60,1.00,Male,Yes,Sat,Dinner,2
0,16.99,1.01,Female,No,Sun,Dinner,2


In [22]:
# UPDATE tips SET tip = tip*2 WHERE tip < 2;
tips.loc[tips["tip"] < 2, "tip"] *= 2
tips.sort_values("tip").head()

,total_bill,tip,sex,smoker,day,time,size
6,8.77,2.0,Male,No,Sun,Dinner,2
26,13.37,2.0,Male,No,Sat,Dinner,2
27,12.69,2.0,Male,No,Sat,Dinner,2
36,16.31,2.0,Male,No,Sat,Dinner,3
61,13.81,2.0,Male,Yes,Sat,Dinner,2


In [23]:
# state before the next change:
tips.sort_values("tip", ascending=False).head()

,total_bill,tip,sex,smoker,day,time,size
170,50.81,10.00,Male,Yes,Sat,Dinner,3
212,48.33,9.00,Male,No,Sat,Dinner,4
23,39.42,7.58,Male,No,Sat,Dinner,4
59,48.27,6.73,Male,No,Sat,Dinner,4
141,34.30,6.70,Male,No,Thur,Lunch,6


In [24]:
# DELETE FROM tips WHERE tip > 9;
tips = tips.loc[tips["tip"] <= 9]
tips.sort_values("tip", ascending=False).head()

,total_bill,tip,sex,smoker,day,time,size
212,48.33,9.00,Male,No,Sat,Dinner,4
23,39.42,7.58,Male,No,Sat,Dinner,4
59,48.27,6.73,Male,No,Sat,Dinner,4
141,34.30,6.70,Male,No,Thur,Lunch,6
183,23.17,6.50,Male,Yes,Sun,Dinner,4


## A first look at a dataset

Pandas comes with the tools we discussed in the chapter on descriptive statistics – the ones that were still handwork with Numpy in the Titanic notebook. The 'broken' CSV from there is read directly by `read_csv`, which understands quotation marks. `info` shows the columns, the data types and how many values are missing: compare this with the Titanic notebook from the chapter on Numpy, where we converted the CSV into a TSV by hand, marked missing values with masks and counted groups with loops.

In [25]:
titanic = pd.read_csv("titanic.csv")
titanic.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


For every numerical column, `describe` gives the count, the mean, the standard deviation (with $n-1$ in the denominator!), the minimum, the quartiles and the maximum:

In [26]:
titanic.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


For categorical columns, `value_counts` counts the levels:

In [27]:
print(titanic["Sex"].value_counts())
print(titanic["Embarked"].value_counts(dropna=False))   # count the missing values too

Sex
male      577
female    314
Name: count, dtype: int64
Embarked
S      644
C      168
Q       77
NaN      2
Name: count, dtype: int64


`isna` spots missing values. You can discard them (`dropna`) or replace them (`fillna`), with the median for instance. How to deal systematically with missing values, outliers, text fields and dates is the subject of the chapter [Data cleaning](data-cleaning.ipynb).

Skewness and kurtosis from the chapter on expected values are built in as well; `kurt` gives the excess kurtosis, that is, the kurtosis minus $3$:

In [28]:
print(titanic.isna().sum())

age = titanic["Age"].fillna(titanic["Age"].median())
print("skewness:", age.skew(), " excess kurtosis:", age.kurt())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
skewness: 0.5102446555756495  excess kurtosis: 0.9938710163801736


Grouped summary measures come from `groupby`, as above:

In [29]:
titanic.groupby("Pclass")["Age"].describe()

,count,mean,std,min,25%,50%,75%,max
Pclass,,,,,,,,
1,186.0,38.233441,14.802856,0.92,27.0,37.0,49.0,80.0
2,173.0,29.877630,14.001077,0.67,23.0,29.0,36.0,70.0
3,355.0,25.140620,12.495398,0.42,18.0,24.0,32.0,74.0
